# Most Impactful Bulls — 2025-26 Impact Board

First board-format post from the ideation playbook: a ranked Game Score leaderboard replacing the court as default canvas.

## Objective

Which Bulls delivered the most per-game box-score impact across the 2025-26 season?

## Data Pull

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parents[1]))

import numpy as np
import pandas as pd

from bulls import data

# Full season of box scores = ~82 API calls (~3 min). Cache locally so reruns are instant.
CACHE_DIR = Path("../../cache")
BOX_CACHE = CACHE_DIR / "box_scores_2025-26.csv"
ROSTER_CACHE = CACHE_DIR / "roster_2025-26.csv"

if BOX_CACHE.exists():
    box = pd.read_csv(BOX_CACHE)
else:
    games = data.get_games(season="2025-26")
    frames = []
    for gid in games["GAME_ID"].unique():
        b = data.get_box_score(gid)
        if not b.empty:
            b["game_id"] = gid
            frames.append(b)
    box = pd.concat(frames, ignore_index=True)
    CACHE_DIR.mkdir(exist_ok=True)
    box.to_csv(BOX_CACHE, index=False)

if ROSTER_CACHE.exists():
    roster = pd.read_csv(ROSTER_CACHE)
else:
    roster = data.get_roster()
    roster.to_csv(ROSTER_CACHE, index=False)

print(f"{box['game_id'].nunique()} games, {len(box)} player-game rows, {len(roster)} on current roster")

## Analysis

In [ ]:
# Block 1: avg Game Score per player (Hollinger), min 20 games played
MIN_GAMES = 20


def parse_minutes(val) -> float:
    """BoxScoreTraditionalV3 minutes come as 'MM:SS' strings ('' for DNP)."""
    if pd.isna(val):
        return 0.0
    s = str(val)
    if ":" not in s:
        return float(s) if s.strip() else 0.0
    mm, ss = s.split(":")
    return int(mm) + int(ss) / 60


def game_score(row) -> float:
    """Hollinger Game Score from a box-score line."""
    return (
        row["points"]
        + 0.4 * row["fieldGoalsMade"]
        - 0.7 * row["fieldGoalsAttempted"]
        - 0.4 * (row["freeThrowsAttempted"] - row["freeThrowsMade"])
        + 0.7 * row["reboundsOffensive"]
        + 0.3 * row["reboundsDefensive"]
        + row["steals"]
        + 0.7 * row["assists"]
        + 0.7 * row["blocks"]
        - 0.4 * row["foulsPersonal"]
        - row["turnovers"]
    )


played = box[box["minutes"].map(parse_minutes) > 0].copy()
played["gmsc"] = played.apply(game_score, axis=1)

board = played.groupby(["personId", "name"], as_index=False).agg(
    games=("game_id", "nunique"),
    gmsc_avg=("gmsc", "mean"),
    pts=("points", "sum"),
    fga=("fieldGoalsAttempted", "sum"),
    fta=("freeThrowsAttempted", "sum"),
)
board["ppg"] = board["pts"] / board["games"]
tsa = board["fga"] + 0.44 * board["fta"]
board["ts_pct"] = np.where(tsa > 0, board["pts"] / (2 * tsa) * 100, 0.0)

board = (
    board[board["games"] >= MIN_GAMES]
    .sort_values("gmsc_avg", ascending=False)
    .reset_index(drop=True)
)
board[["name", "games", "gmsc_avg", "ppg", "ts_pct"]].round(1)

In [ ]:
# Block 2: current-roster view (traded/departed players drop out)
roster_ids = set(roster["player_id"].astype(int))
board_current = board[board["personId"].astype(int).isin(roster_ids)].reset_index(drop=True)

departed = board[~board["personId"].astype(int).isin(roster_ids)]
print("Departed since the season:", ", ".join(departed["name"]))
board_current[["name", "games", "gmsc_avg", "ppg", "ts_pct"]].round(1)

## Output

In [ ]:
# Board graphic: house style (Playfair title, red kicker) + Basketball University mechanics
# (ranked rows, headshots, shared-scale bars, visible threshold). Notebook-resident for now;
# extract into bulls/graphics once the board format proves repeatable.
import matplotlib.pyplot as plt

from bulls.data.fetch import get_player_headshot
from bulls.graphics.feed import (
    DEFAULT_DPI,
    INSTAGRAM_FEED_HEIGHT_PX,
    INSTAGRAM_FEED_WIDTH_PX,
    _fp_body,
    _fp_title,
    _make_circular_headshot,
    save_feed_post,
)

INK, MUTED, FAINT, RULE, RED = "#1A1A1A", "#777777", "#AAAAAA", "#DDDDDD", "#CE1141"
TOP_N = 10


def build_impact_board(
    board: pd.DataFrame,
    title: str = "Most Impactful Bulls",
    subtitle: str = "Chicago Bulls | 2025-26 Season",
    kicker: str = f"Ranked by Avg. Game Score | Min. {MIN_GAMES} Games",
) -> plt.Figure:
    rows = board.head(TOP_N)

    figsize = (INSTAGRAM_FEED_WIDTH_PX / DEFAULT_DPI, INSTAGRAM_FEED_HEIGHT_PX / DEFAULT_DPI)
    fig = plt.figure(figsize=figsize, facecolor="#FFFFFF")
    ax = fig.add_axes([0.0, 0.0, 1.0, 1.0])
    ax.set_facecolor("#FFFFFF")
    ax.set_xlim(0, INSTAGRAM_FEED_WIDTH_PX)
    ax.set_ylim(0, INSTAGRAM_FEED_HEIGHT_PX)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    top = INSTAGRAM_FEED_HEIGHT_PX
    ax.text(60, top - 70, title, ha="left", va="top",
            fontsize=44, color=INK, fontproperties=_fp_title(weight="bold"))
    ax.text(60, top - 185, subtitle, ha="left", va="top",
            fontsize=18, color=MUTED, fontproperties=_fp_body(weight="medium"))
    ax.text(60, top - 235, kicker, ha="left", va="top",
            fontsize=14, color=RED, style="italic", fontproperties=_fp_body(weight="medium"))

    x_rank, x_photo, x_name = 85, 150, 205
    bar_x0, bar_x1 = 435, 720
    x_gmsc = bar_x1 + 18
    x_ppg, x_ts, x_gp = 870, 960, 1035

    header_y = top - 300
    ax.text(x_name, header_y, "PLAYER", ha="left", va="center",
            fontsize=11, color=MUTED, fontproperties=_fp_body(weight="bold"))
    ax.text(bar_x0, header_y, "AVG. GAME SCORE", ha="left", va="center",
            fontsize=11, color=MUTED, fontproperties=_fp_body(weight="bold"))
    for x, label in ((x_ppg, "PPG"), (x_ts, "TS%"), (x_gp, "GP")):
        ax.text(x, header_y, label, ha="center", va="center",
                fontsize=11, color=MUTED, fontproperties=_fp_body(weight="bold"))

    row_top = header_y - 30
    row_h, bar_h = 92, 24
    max_gmsc = max(rows["gmsc_avg"].max(), 1.0)

    for i, row in rows.iterrows():
        y = row_top - i * row_h - row_h / 2

        if i > 0:
            ax.plot([60, 1035], [y + row_h / 2, y + row_h / 2],
                    color=RULE, lw=0.8, solid_capstyle="butt")

        ax.text(x_rank, y, str(i + 1), ha="right", va="center",
                fontsize=20, color=FAINT, fontproperties=_fp_body(weight="bold"))

        headshot_path = get_player_headshot(int(row["personId"]))
        if headshot_path and Path(headshot_path).exists():
            circ = _make_circular_headshot(Path(headshot_path))
            if circ is not None:
                r = 34
                ax.imshow(circ, extent=[x_photo - r, x_photo + r, y - r, y + r],
                          zorder=3, interpolation="bilinear")

        name = row["name"]
        if len(name) > 16:
            first, rest = name.split(" ", 1)
            name = f"{first[0]}. {rest}"
        ax.text(x_name, y, name, ha="left", va="center",
                fontsize=15, color=INK, fontproperties=_fp_body(weight="bold"))

        w = (row["gmsc_avg"] / max_gmsc) * (bar_x1 - bar_x0)
        ax.add_patch(plt.Rectangle((bar_x0, y - bar_h / 2), w, bar_h,
                                   facecolor=RED, edgecolor="none", zorder=2))
        ax.text(x_gmsc, y, f"{row['gmsc_avg']:.1f}", ha="left", va="center",
                fontsize=17, color=INK, fontproperties=_fp_body(weight="bold"))

        for x, val in ((x_ppg, f"{row['ppg']:.1f}"), (x_ts, f"{row['ts_pct']:.1f}"),
                       (x_gp, f"{int(row['games'])}")):
            ax.text(x, y, val, ha="center", va="center",
                    fontsize=14, color=MUTED, fontproperties=_fp_body(weight="medium"))

    ax.text(60, 62,
            "Game Score (Hollinger): one box-score number covering scoring, "
            "rebounds, assists, defense, turnovers.",
            ha="left", va="bottom", fontsize=8.5, color=FAINT, fontproperties=_fp_body())
    ax.text(60, 40, "Data via NBA.com/Stats", ha="left", va="bottom",
            fontsize=8.5, color=FAINT, fontproperties=_fp_body())

    return fig


OUTPUT_DIR = Path("../../output/feed")

fig_all = build_impact_board(board)
save_feed_post(fig_all, OUTPUT_DIR / "2026-07-03-impact-board-season.png")

fig_current = build_impact_board(
    board_current, subtitle="Chicago Bulls | 2025-26 Season | Current Roster")
save_feed_post(fig_current, OUTPUT_DIR / "2026-07-03-impact-board-season-current-roster.png")

fig_all

## Takeaways

- Josh Giddey was the season's clearest per-game engine: 16.2 avg Game Score, nearly three clear of anyone still on the roster.
- Three of the top six impact players (Vučević, Coby White, Dosunmu) have since departed — the current-roster board thins out fast after Giddey and Tre Jones.
- Availability splits the board: only Buzelis (77 GP) and Tre Jones (65) cleared 60 games in the top 10, while Sexton and White ranked high on just 26-29 appearances.

## Next Question

Does the ranking change when sorted by total Game Score (per-game impact × games played) — who actually carried the season's minutes?